# Part 2 supplementary notebook — historical data-foundation command review

> **Primary reproducible notebook:** `MemberC_Phases_1_3_Full_Reproduction.ipynb`

This smaller notebook is retained as a historical/supplementary review of the original Part 2 scripts. It is **not** the preferred one-click reproduction entry point because one cell contains a machine-specific path to the large historical website CSV.

The scientific purpose is to show the same three internal stages used by Part 2:

- canonical label/provenance reconstruction;
- duplicate-safe enrichment;
- author/thread/exact-content grouped splitting.

The final project recall floors (Low 75%, Moderate 50%, High 50%, Very High 75%) belong to later model evaluation; they are internal project targets, not clinical standards.


In [1]:
from pathlib import Path
import csv, gzip, json
from collections import Counter

ROOT = Path('..').resolve()
OUTPUTS = ROOT / 'outputs'
report = json.loads((OUTPUTS / 'phase1_report.json').read_text(encoding='utf-8'))
print(json.dumps(report['counts'], ensure_ascii=False, indent=2))

{
  "original_dual_rows": 2000,
  "historical_fusion_rows": 2229,
  "original_dual_in_historical_fusion": 1607,
  "original_dual_removed_by_historical_undersampling": 393,
  "recovered_original_p1_high_selected": 622,
  "active_rows": 3000,
  "active_dual_rows": 1000,
  "active_p2_calibrated_rows": 2000,
  "duplicate_ids_across_labeling_rounds": 5,
  "canonical_unique_posts": 5617,
  "annotation_history_rows": 10622
}


In [2]:
with gzip.open(OUTPUTS / 'canonical_labels.csv.gz', 'rt', encoding='utf-8-sig', newline='') as f:
    canonical = list(csv.DictReader(f))

print('Canonical unique posts:', len(canonical))
print('Label sources:', dict(Counter(r['label_source'] for r in canonical)))
print('Clinical classes:', dict(Counter(r['clinical_class'] for r in canonical)))
print('Repeated IDs resolved:', sum(r['duplicate_id_across_rounds'] == 'True' for r in canonical))
print('Rows in exact duplicate-content groups:', sum(r['exact_duplicate_content'] == 'True' for r in canonical))

Canonical unique posts: 5617
Label sources: {'active_dual': 1000, 'active_p2_calibrated': 1995, 'original_p1_high_selected': 622, 'original_dual': 2000}
Clinical classes: {'Moderate': 1222, 'Low': 3029, 'High': 974, 'Very High': 392}
Repeated IDs resolved: 5
Rows in exact duplicate-content groups: 30


## Historical Phase 1 result — why the calibration/provenance logic matters

Dual-annotated rows use the historical weighted consensus `(1.7*Person1 + Person2)/2.7`.

For Person-2-only active-learning rows, the project first estimates the missing Person-1 score from the dual-labeled overlap using ordinary least squares, then applies the same consensus formula. OLS learns a slope/intercept that minimizes squared calibration residuals on the overlap.

This produces a lower-confidence continuous training label; it does **not** make a single-rater estimate equivalent to clinical ground truth.

Repeated IDs are canonicalized to one modeling row while all annotation events remain in `annotation_history.csv.gz`. Exact duplicate text receives a shared hash so the split stage can keep duplicates together.


In [3]:
cal = report['active_calibration']
print('Person2 -> Person1 slope:', cal['person2_to_person1_slope'])
print('Person2 -> Person1 intercept:', cal['person2_to_person1_intercept'])
print('5-fold CV MAE to weighted consensus:', cal['cv_metrics']['linear_p2_to_p1_then_weighted']['mae_to_consensus'])

Person2 -> Person1 slope: 0.6459265242978497
Person2 -> Person1 intercept: 0.5037576412178764
5-fold CV MAE to weighted consensus: 0.3913152089218337


## Historical Phase 2 — duplicate-safe local enrichment

This command demonstrates the raw-data enrichment stage. The large cleaned website file is not portable in this small notebook because the historical cell contains a local Windows path.

Use the **primary Part 2 full reproduction notebook** for a portable packaged workflow.

Algorithmically, Phase 2 must not perform an unchecked many-to-many ID merge. It streams the website export, uses `unique_post_id` as the primary key, and resolves repeated IDs with normalized-content evidence. Ambiguous rows should be excluded rather than guessed.

The old strict command is useful as a diagnostic because it stops on ambiguity. The accepted final project intentionally excludes the two unresolved ambiguous matches and retains 5,615 enriched rows.


In [4]:
RAW_360K = Path(r'C:\Education\term6-private\DS\Project\New folder (3)\data\cleaned_full_ninisite_stress_proxy3.csv')
phase2_command = [
    'python', str(ROOT / 'src' / 'phase2_extract_enriched.py'),
    '--raw', str(RAW_360K),
    '--labels', str(OUTPUTS / 'canonical_labels.csv.gz'),
    '--output', str(OUTPUTS / 'canonical_labels_enriched.csv.gz'),
    '--audit', str(OUTPUTS / 'phase2_match_audit.csv'),
    '--report', str(OUTPUTS / 'phase2_report.json'),
    '--strict',
]
print(' '.join(f'"{x}"' if ' ' in x else x for x in phase2_command))
print()
print('Not executed here because the 176 MB raw file is not uploaded.')

python /mnt/data/phase1_3_pipeline/src/phase2_extract_enriched.py --raw "C:\Education\term6-private\DS\Project\New folder (3)\data\cleaned_full_ninisite_stress_proxy3.csv" --labels /mnt/data/phase1_3_pipeline/outputs/canonical_labels.csv.gz --output /mnt/data/phase1_3_pipeline/outputs/canonical_labels_enriched.csv.gz --audit /mnt/data/phase1_3_pipeline/outputs/phase2_match_audit.csv --report /mnt/data/phase1_3_pipeline/outputs/phase2_report.json --strict

Not executed here because the 176 MB raw file is not uploaded.


## Historical Phase 3 — connected-component split rather than random rows

The split builder links posts through author, thread, and exact duplicate content. Union-Find/Disjoint Set Union converts these relations into connected components, including transitive connections.

Whole components are assigned together so a model cannot see the same author/conversation/duplicate text on both sides of the evaluation boundary.

The final primary Part 2 notebook continues beyond this historical preview by creating official modeling roles, embargo rows, five grouped OOF folds, and the Member 1/Member 2 handoffs.


In [5]:
phase3_command = [
    'python', str(ROOT / 'src' / 'phase3_build_split_manifest.py'),
    '--enriched', str(OUTPUTS / 'canonical_labels_enriched.csv.gz'),
    '--output', str(OUTPUTS / 'split_manifest.csv'),
    '--report', str(OUTPUTS / 'phase3_split_report.json'),
]
print(' '.join(f'"{x}"' if ' ' in x else x for x in phase3_command))
print()
print('Run after Phase 2. The final manifest is intentionally not fabricated without author/thread metadata.')

python /mnt/data/phase1_3_pipeline/src/phase3_build_split_manifest.py --enriched /mnt/data/phase1_3_pipeline/outputs/canonical_labels_enriched.csv.gz --output /mnt/data/phase1_3_pipeline/outputs/split_manifest.csv --report /mnt/data/phase1_3_pipeline/outputs/phase3_split_report.json

Run after Phase 2. The final manifest is intentionally not fabricated without author/thread metadata.


## Supplementary deliverables

This notebook documents the historical scripts and their intermediate artifacts. The main Part 2 reproduction notebook adds stronger assertions, the accepted frozen-enrichment fallback, final role cleanup, OOF folds, and weights.

The key methodological lesson is that label provenance and split leakage must be resolved **before** model training.
